In [2]:
import pandas as pd
import numpy as np
from fomoz.color_selection import (
    Measure,
    Optimize,
    SelectBackground
)

In [ ]:
# Load the reference sample, having the entries
# 'g', 'r', 'i', 'redshift'
ref_sample = pd.read_csv('reference.csv')

# Define the z_l points and bands
# used for the calibration of the selection
z_l = [0.4, 0.5, 0.6, 0.7]
bands = ['g', 'r', 'i']

# c_i and s_i samples
c_samples = np.linspace(-5, 5, 100)
s_samples = np.linspace(0.3, 1.7, 30)

# Measure C, P, and F for all color conditions
measure = Measure(
    ref_sample, bands, z_l,
    c_samples=c_samples,
    s_samples=s_samples,
    z_gal_string='redshift',  # redshift string
    missing_data=-99,  # non-detection value
)
measure.process()

In [ ]:
# Get the optimal combination of color conditions
P_min = 0.97  # global purity threshold

optim = Optimize(measure)
optim.process(
    P_min,
    start_purity=0.999,
    step=0.0001
)

# Plot the global completeness, purity,
# and foreground failure rate
optim.plot()

In [ ]:
# Load the test sample
# (it's the reference sample in this example)
test_sample = ref_sample

# New test z_l points
zl_new = np.arange(min(z_l), max(z_l), 0.005)

# SelectBackground instance, providing
# the Optimize object in input. Alternatively,
# the file written with write_optimal_selection()
# can be provided in input
sel = SelectBackground()
sel.load_parameters(optim)

# Galaxy selection.
# mask is a list of boolean arrays.
# The i-th array corresponds to zl_new[i].
sel.process(zl_new, test_sample)
mask = sel.mask